# Unit 2: Audio Applications with Pipelines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

**Course**: [HuggingFace Audio Transformers Course - Unit 2](https://huggingface.co/learn/audio-course/chapter2)

**Topics covered:**
- Audio classification with `pipeline()`
- Automatic speech recognition (ASR) with `pipeline()`
- Text-to-speech generation with `pipeline()`
- Exploring pre-trained models on the Hub

## Setup

In [ ]:
!pip install -q transformers datasets librosa soundfile torch torchaudio

## 1. Audio Classification

Classify audio into categories: speech commands, environmental sounds, music genres, etc.

In [ ]:
from transformers import pipeline
import IPython.display as ipd

# Audio classification pipeline
classifier = pipeline("audio-classification", model="MIT/ast-finetuned-audioset-10-10-0.4593")

# Classify a sample audio
import librosa
audio_array, sr = librosa.load(librosa.ex('trumpet'), sr=16000)

result = classifier(audio_array)
for item in result:
    print(f"{item['label']}: {item['score']:.4f}")

### Keyword Spotting

Detecting specific words or commands in speech.

In [ ]:
from datasets import load_dataset

# Load a speech commands dataset
speech_commands = load_dataset(
    "speech_commands", "v0.02", split="validation", streaming=True, trust_remote_code=True
)

example = next(iter(speech_commands))
print(f"Label: {example['label']}")
print(f"Sampling rate: {example['audio']['sampling_rate']}")

ipd.Audio(example['audio']['array'], rate=example['audio']['sampling_rate'])

## 2. Automatic Speech Recognition (ASR)

Convert spoken speech to text.

In [ ]:
# ASR pipeline with Whisper
asr = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")

# Load a speech sample
minds14 = load_dataset("PolyAI/minds14", name="en-US", split="train", trust_remote_code=True)
sample = minds14[0]

print(f"Ground truth: {sample.get('transcription', 'N/A')}")

result = asr(sample["audio"]["array"])
print(f"Prediction:   {result['text']}")

In [ ]:
# ASR with timestamps (long-form audio)
result_with_timestamps = asr(
    sample["audio"]["array"],
    return_timestamps=True
)
print(result_with_timestamps)

## 3. Text-to-Speech (TTS)

Generate spoken audio from text input.

In [ ]:
# TTS pipeline
tts = pipeline("text-to-speech", model="microsoft/speecht5_tts")

# We need speaker embeddings for SpeechT5
from datasets import load_dataset
import torch

embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)

result = tts("Hello! Welcome to the HuggingFace audio course.", forward_params={"speaker_embeddings": speaker_embedding})

print(f"Output sampling rate: {result['sampling_rate']}")
print(f"Audio shape: {result['audio'].shape}")

ipd.Audio(result["audio"], rate=result["sampling_rate"])

## 4. Exploring Models on the Hub

Tips for finding models:
- Filter by task: `audio-classification`, `automatic-speech-recognition`, `text-to-speech`
- Check model cards for dataset, language, and performance details
- Look at download counts and community feedback
- Use `pipeline(task, model=model_id)` to try any model instantly

In [ ]:
from huggingface_hub import list_models

# Find popular ASR models
models = list_models(task="automatic-speech-recognition", sort="downloads", direction=-1, limit=5)

print("Top 5 ASR models by downloads:")
for model in models:
    print(f"  {model.id} ({model.downloads:,} downloads)")

## Key Takeaways

- `pipeline()` provides a high-level API for audio tasks with minimal code
- Audio classification: categorize sounds, detect keywords, classify music
- ASR: transcribe speech to text, supports timestamps for long audio
- TTS: generate natural-sounding speech from text
- The HuggingFace Hub has thousands of pre-trained audio models ready to use

**Next**: [Unit 3 - Transformer Architectures for Audio](../unit_3/03_transformer_architectures.ipynb)